## Установка и импорт библиотек

In [16]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.3.2/spark-3.3.2-bin-hadoop3.tgz
!tar xf spark-3.3.2-bin-hadoop3.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.3.2-bin-hadoop3"

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, explode, count, row_number, desc, trim, lower
from pyspark.sql.window import Window
spark = SparkSession.builder \
    .appName("StackOverflow Language Popularity") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.jars.packages", "com.databricks:spark-xml_2.12:0.16.0") \
    .getOrCreate()

spark = SparkSession.builder \
    .appName("StackOverflow Language Popularity") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

import os
sample_path = "sample_data/posts_sample.xml"
df_posts = spark.read.format("xml") \
        .option("rowTag", "row") \
        .option("attributePrefix", "") \
        .load(sample_path)
# Загрузка списка языков
langs_path = "sample_data/programming-languages.csv"
if os.path.exists(langs_path):
    df_langs = spark.read.csv(langs_path, header=True, inferSchema=True)
    print(f"Языков в справочнике: {df_langs.count()}")



Языков в справочнике: 700


## Очистка и подготовка данных

In [17]:
df_filtered = df_posts.select(
    col("Id").alias("post_id"),
    col("Tags"),
    col("CreationDate")
).filter(
    (year("CreationDate") >= 2010) & (year("CreationDate") <= 2020)
)

df_with_year = df_filtered.withColumn("year", year("CreationDate"))
from pyspark.sql.functions import split, regexp_replace, trim

df_tags_exploded = df_with_year \
    .withColumn("Tags", regexp_replace("Tags", "><", ",")) \
    .withColumn("Tags", regexp_replace("Tags", "<", "")) \
    .withColumn("Tags", regexp_replace("Tags", ">", "")) \
    .withColumn("Tags", split("Tags", ",")) \
    .withColumn("tag", explode("Tags")) \
    .select("year", "tag")

# Убираем пустые теги
df_tags_exploded = df_tags_exploded.filter(trim("tag") != "")

df_langs_clean = df_langs.withColumn("lang_lower", trim(lower(col("name"))))

# Джойним теги со справочником языков
df_tagged_langs = df_tags_exploded.join(
    df_langs_clean,
    df_tags_exploded["tag"] == df_langs_clean["lang_lower"],
    "inner"
).select("year", col("tag").alias("language"))

print("Примеры тегов, распознанных как языки:")
df_tagged_langs.show()


Примеры тегов, распознанных как языки:
+----+-----------+
|year|   language|
+----+-----------+
|2010|       java|
|2010|        php|
|2010|       ruby|
|2010|          c|
|2010|        php|
|2010|     python|
|2010| javascript|
|2010|applescript|
|2010|        php|
|2010|        php|
|2010| javascript|
|2010|        sed|
|2010|     python|
|2010|       java|
|2010|       ruby|
|2010|objective-c|
|2010| javascript|
|2010|          r|
|2010|        php|
|2010| javascript|
+----+-----------+
only showing top 20 rows



## Формирование отчета


In [18]:
# Группируем по году и языку, считаем количество упоминаний
df_lang_counts = df_tagged_langs \
    .groupBy("year", "language") \
    .agg(count("*").alias("mentions"))

# Оконная функция для ранжирования в каждом году
window_spec = Window.partitionBy("year").orderBy(desc("mentions"))

df_ranked = df_lang_counts \
    .withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") <= 10)

result = df_ranked.select("year", "language", "mentions", "rank") \
    .orderBy("year", "rank")

print("Топ-10 языков по годам:")
result.show(20, truncate=False)

output_path = "stackoverflow_lang_top10"
result.write.mode("overwrite").parquet(output_path)

# Проверяем, что сохранилось
df_loaded = spark.read.parquet(output_path)
print(f"\nОтчёт сохранён. Количество строк: {df_loaded.count()}")
df_loaded.printSchema()

print(f"\nОтчёт сохранён в формате Parquet в папке: {os.path.abspath(output_path)}")
print("Файлы отчёта:")
!ls -la stackoverflow_lang_top10/

print("\nПример чтения Parquet файла:")
sample_parquet = spark.read.parquet(f"{output_path}/part-00000-*.parquet")
sample_parquet.show(5, truncate=False)

spark.stop()


Топ-10 языков по годам:
+----+-----------+--------+----+
|year|language   |mentions|rank|
+----+-----------+--------+----+
|2010|java       |52      |1   |
|2010|php        |46      |2   |
|2010|javascript |44      |3   |
|2010|python     |26      |4   |
|2010|objective-c|23      |5   |
|2010|c          |20      |6   |
|2010|ruby       |12      |7   |
|2010|delphi     |8       |8   |
|2010|applescript|3       |9   |
|2010|r          |3       |10  |
|2011|php        |102     |1   |
|2011|java       |93      |2   |
|2011|javascript |83      |3   |
|2011|python     |37      |4   |
|2011|objective-c|34      |5   |
|2011|c          |24      |6   |
|2011|ruby       |20      |7   |
|2011|perl       |9       |8   |
|2011|delphi     |8       |9   |
|2011|bash       |7       |10  |
+----+-----------+--------+----+
only showing top 20 rows


Отчёт сохранён. Количество строк: 100
root
 |-- year: integer (nullable = true)
 |-- language: string (nullable = true)
 |-- mentions: long (nullable = true)